# 🦷 Dental 3D Diş Segmentasyonu — Colab A100 Eğitimi

Bu notebook iki modeli **aynı hasta bazlı bölme** üzerinde eğitir ve test setinde karşılaştırır:

1. **DGCNN** (yerel komşuluk, ana model)
2. **PointNet** (karşılaştırma / baseline)

Tüm çıktılar (checkpoint, eğitim eğrileri, test metrikleri, örnek mesh'ler) Google Drive'a yazılır;
Colab bağlantısı koparsa **"Kaldığı yerden devam"** hücresini çalıştırmanız yeterli.

**Çalıştırmadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Ayarlar
Drive'daki yolları kendi düzeninize göre değiştirin.

In [ ]:
# --- KOD KAYNAGI ---
# "zip": Drive'a yuklediginiz DentalScannerProject_code.zip'ten ac (varsayilan)
# "git": GitHub deposundan klonla (guncel kod push edildiyse)
KOD_KAYNAGI = "zip"
GIT_URL     = "https://github.com/Emirhan-Kar/Dental3DScanner.git"
KOD_ZIP     = "/content/drive/MyDrive/DentalScanner/DentalScannerProject_code.zip"

# --- VERI ---
# Icinde 3DTeethData/data_part_*/... bulunan zip (README'deki Dental_3DTeeth_Dataset.zip de olur)
VERI_ZIP    = "/content/drive/MyDrive/DentalScanner/Dental_3DTeeth_Dataset.zip"

# --- CIKTILAR (Drive'da kalici) ---
CIKTI_KOK   = "/content/drive/MyDrive/DentalScanner/runs"

# --- EGITIM ---
DGCNN_EPOCH    = 100
POINTNET_EPOCH = 100
NOKTA          = 8192
BATCH          = 8
ISCI           = 8

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(CIKTI_KOK, exist_ok=True)

## 2. Kod ve bağımlılıklar

In [ ]:
%cd /content
if KOD_KAYNAGI == "git":
    !rm -rf /content/proj && git clone -q {GIT_URL} /content/proj
else:
    !rm -rf /content/proj && mkdir -p /content/proj && unzip -q "{KOD_ZIP}" -d /content/proj
    # zip tek bir ust klasor iceriyorsa onu kok yap
    import glob, shutil
    alt = [d for d in glob.glob('/content/proj/*') if os.path.isdir(d)]
    if len(alt) == 1 and os.path.exists(os.path.join(alt[0], 'main.py')):
        for f in os.listdir(alt[0]): shutil.move(os.path.join(alt[0], f), '/content/proj')
%cd /content/proj
!pip -q install open3d
!ls

## 3. Veri setini yerel diske aç
Drive'dan doğrudan okumak çok yavaştır; zip yerel diske kopyalanıp sadece `3DTeethData` açılır (~10 dk).

In [ ]:
import zipfile, time
t = time.time()
!cp "{VERI_ZIP}" /content/veri.zip
z = zipfile.ZipFile('/content/veri.zip')
uyeler = [n for n in z.namelist() if '3DTeethData/' in n and (n.endswith('.obj') or n.endswith('.json'))]
onek = uyeler[0][:uyeler[0].index('3DTeethData/')]
print(f"{len(uyeler)} dosya, zip ici onek: '{onek}'")
z.extractall('/content/data_tmp', members=uyeler)
os.makedirs('/content/proj/data', exist_ok=True)
!rm -rf /content/proj/data/3DTeethData
shutil_src = os.path.join('/content/data_tmp', onek, '3DTeethData')
os.rename(shutil_src, '/content/proj/data/3DTeethData')
!rm -f /content/veri.zip
print(f"Bitti: {time.time()-t:.0f} sn")
!ls /content/proj/data/3DTeethData
!find /content/proj/data/3DTeethData -name "*.obj" | wc -l

## 4. Hızlı kontrol (testler + tek vaka)

In [ ]:
!python -m pytest tests -q 2>&1 | tail -3

## 5. DGCNN eğitimi (ana model)
- Hasta bazlı bölme `split.json` olarak kaydedilir (%80 eğitim / %10 doğrulama / %10 test).
- İlk epoch mesh'leri `.npz` önbelleğine çevirdiği için daha yavaştır; sonrakiler hızlıdır.
- Her epoch `last.pth`, en iyi doğrulama mIoU'sunda `best.pth` Drive'a yazılır.
- Eğitim bitince en iyi model test setinde **tüm köşeler** üzerinde değerlendirilir → `eval_test/`.

In [ ]:
DGCNN_DIR = f"{CIKTI_KOK}/dgcnn"
!python ai_model/train.py --model dgcnn --epochs {DGCNN_EPOCH} --num-points {NOKTA} --batch-size {BATCH} \
    --workers {ISCI} --cache-dir /content/cache --out "{DGCNN_DIR}"

### 5b. Bağlantı koptuysa: kaldığı yerden devam
(Önce 1-3. hücreleri tekrar çalıştırın.)

In [ ]:
!python ai_model/train.py --model dgcnn --epochs {DGCNN_EPOCH} --num-points {NOKTA} --batch-size {BATCH} \
    --workers {ISCI} --cache-dir /content/cache --out "{DGCNN_DIR}" --resume "{DGCNN_DIR}/last.pth"

## 6. PointNet eğitimi (karşılaştırma)
Adil karşılaştırma için DGCNN'in `split.json` dosyası kopyalanır → aynı test hastaları.

In [ ]:
POINTNET_DIR = f"{CIKTI_KOK}/pointnet"
os.makedirs(POINTNET_DIR, exist_ok=True)
!cp "{DGCNN_DIR}/split.json" "{POINTNET_DIR}/split.json"
!python ai_model/train.py --model pointnet --epochs {POINTNET_EPOCH} --num-points {NOKTA} --batch-size 16 \
    --workers {ISCI} --cache-dir /content/cache --out "{POINTNET_DIR}"

## 7. Eski model (best_dental_model.pth) da aynı test setinde ölçülsün
Raporda "önceki sürüm → yeni sürüm" karşılaştırması için.

In [ ]:
!python ai_model/evaluate.py --checkpoint ai_model/best_dental_model.pth --split "{DGCNN_DIR}/split.json" \
    --num-points 4096 --out "{CIKTI_KOK}/old_pointnet/eval_test"

## 8. Sonuçlar

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
satirlar = []
for ad, yol in [("Eski PointNet", f"{CIKTI_KOK}/old_pointnet/eval_test"),
                ("PointNet (yeni egitim)", f"{POINTNET_DIR}/eval_test"),
                ("DGCNN", f"{DGCNN_DIR}/eval_test")]:
    f = f"{yol}/test_metrics.json"
    if os.path.exists(f):
        m = json.load(open(f))
        satirlar.append({"Model": ad, "Vaka": m["num_cases"], "Dogruluk": m["accuracy"],
                         "mIoU (disler)": m["mIoU_teeth"], "mIoU (tum)": m["mIoU_all"], "Dice": m["dice"],
                         "Dis tanima (IoU>0.5)": m["tooth_identification_rate_iou50"]})
tablo = pd.DataFrame(satirlar).set_index("Model")
tablo.to_csv(f"{CIKTI_KOK}/model_karsilastirma.csv")
display(tablo.style.format("{:.3f}", subset=tablo.columns[1:]))

In [ ]:
for yol in [f"{DGCNN_DIR}/learning_curves.png", f"{DGCNN_DIR}/eval_test/per_tooth_iou.png",
            f"{POINTNET_DIR}/eval_test/per_tooth_iou.png"]:
    if os.path.exists(yol): display(Image(yol))

## 9. En iyi modeli projeye al
Bu dosyayı indirip bilgisayarınızda `ai_model/best_dental_model.pth` yerine koyun (eskisini yedekleyin).

In [ ]:
!cp "{DGCNN_DIR}/best.pth" "{CIKTI_KOK}/best_dental_model_dgcnn.pth"
print("Drive'da:", f"{CIKTI_KOK}/best_dental_model_dgcnn.pth")